## Article ↔ Document Group Analysis

Identifies the many-to-many relationship between articles and document groups: articles that appear in more than one reprint group.

- **Input:** `../_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv`
- **Output:** `multi_group_articles.csv` (this folder)

In [ ]:
import pandas as pd

INFILE  = "../_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = "multi_group_articles.csv"

### Which articles belong to more than one document group?

In [ ]:
df = pd.read_csv(INFILE, dtype=str).fillna("")

# Use compound_object rows only — one row per article per document group
# (excludes image child rows so each article-group pair is counted once)
compound = df[df["image_display_template"].str.lower().str.strip() == "compound_object"].copy()

article_groups = (
    compound.groupby("article_id")["group_reprint_id"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .reset_index()
    .rename(columns={"group_reprint_id": "document_groups"})
)
article_groups["num_groups"] = article_groups["document_groups"].apply(len)

multi = article_groups[article_groups["num_groups"] > 1].sort_values("num_groups", ascending=False)

print(f"Total articles: {len(article_groups)}")
print(f"Articles in multiple document groups: {len(multi)}\n")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
print(multi[["article_id", "num_groups", "document_groups"]].to_string(index=False))

### Export results to CSV

In [ ]:
# Flatten document_groups list to semicolon-separated string for CSV storage
export = multi.copy()
export["document_groups"] = export["document_groups"].apply(lambda x: "; ".join(x))

export.to_csv(OUTFILE, index=False)
print(f"Saved {len(export)} rows to {OUTFILE}")